In [ ]:
import sys
import subprocess
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
import ipywidgets as widgets
from IPython.display import display, Image

# Add project root to path
sys.path.append(str(Path('..').resolve()))

from dendritic_modeling.networks.factories.vision_units import rf_on_center, rf_off_center, rf_gabor
from dendritic_modeling.networks.layers.dendritic_conv2d import DendriteConv2d
from dendritic_modeling.metrics.visual import (
    orientation_selectivity_index, 
    direction_selectivity_index,
    size_tuning_index,
    length_selectivity_index
)

plt.style.use('default')
torch.manual_seed(42)
np.random.seed(42)


In [ ]:
# Generate target receptive fields
targets = {
    'ON-center (Retinal)': rf_on_center(size=25),
    'OFF-center (Retinal)': rf_off_center(size=25),
    'Gabor 0° (V1 Simple)': rf_gabor(size=31, theta=np.deg2rad(0)),
    'Gabor 45° (V1 Simple)': rf_gabor(size=31, theta=np.deg2rad(45)),
    'Gabor 90° (V1 Simple)': rf_gabor(size=31, theta=np.deg2rad(90)),
    'Gabor 135° (V1 Simple)': rf_gabor(size=31, theta=np.deg2rad(135)),
}

# Visualize all target RFs
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for i, (name, target_rf) in enumerate(targets.items()):
    vmax = np.abs(target_rf).max()
    im = axes[i].imshow(target_rf, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[i].set_title(name, fontsize=11)
    axes[i].axis('off')
    plt.colorbar(im, ax=axes[i], fraction=0.046)

plt.suptitle('Target Receptive Fields from Visual System Literature', fontsize=14)
plt.tight_layout()
plt.show()

print("Target RF properties:")
for name, target_rf in targets.items():
    print(f"  {name}: shape {target_rf.shape}, range [{target_rf.min():.3f}, {target_rf.max():.3f}]")


In [ ]:
def fit_dendritic_neuron(target_rf, steps=1000, lr=0.1, verbose=True):
    """Fit a dendritic neuron to match target RF using MSE loss on E-I difference."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    target = torch.tensor(target_rf, dtype=torch.float32, device=device)
    
    # Create trainable dendritic layer
    layer = DendriteConv2d(
        in_channels=1, 
        out_channels=1, 
        kernel_size=target.shape[0], 
        padding=target.shape[0]//2, 
        bias=False
    ).to(device)
    
    optimizer = torch.optim.Adam(layer.parameters(), lr=lr)
    losses = []
    
    for step in range(steps):
        optimizer.zero_grad()
        
        # Get E and I weights, compute difference
        E = layer.weight_E[0, 0]  # Remove batch/channel dims
        I = layer.weight_I[0, 0]
        diff = E - I
        
        # MSE loss between E-I difference and target RF
        loss = torch.nn.functional.mse_loss(diff, target)
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        
        if verbose and (step % 200 == 0 or step == steps - 1):
            print(f"Step {step:4d}: Loss = {loss.item():.6f}")
    
    # Return final weights and training history
    with torch.no_grad():
        final_E = layer.weight_E[0, 0].cpu().numpy()
        final_I = layer.weight_I[0, 0].cpu().numpy()
        final_diff = final_E - final_I
    
    return final_E, final_I, final_diff, losses

# Example: Fit to a Gabor
print("Fitting dendritic neuron to 45° Gabor target...")
target_rf = rf_gabor(size=31, theta=np.deg2rad(45))
E_weights, I_weights, EI_diff, losses = fit_dendritic_neuron(target_rf, steps=800, lr=0.05)

# Visualize results
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Target
vmax = np.abs(target_rf).max()
axes[0].imshow(target_rf, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[0].set_title('Target RF (45° Gabor)')
axes[0].axis('off')

# Learned E weights
axes[1].imshow(E_weights, cmap='Reds', vmin=0, vmax=E_weights.max())
axes[1].set_title('Learned E weights')
axes[1].axis('off')

# Learned I weights  
axes[2].imshow(I_weights, cmap='Blues', vmin=0, vmax=I_weights.max())
axes[2].set_title('Learned I weights')
axes[2].axis('off')

# E-I difference (final RF)
axes[3].imshow(EI_diff, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[3].set_title('Learned RF (E-I)')
axes[3].axis('off')

plt.tight_layout()
plt.show()

# Training curve
plt.figure(figsize=(8, 4))
plt.plot(losses, 'b-', linewidth=2)
plt.xlabel('Training Step')
plt.ylabel('MSE Loss')
plt.title('Training Convergence')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Final MSE: {losses[-1]:.6f}")
print(f"Correlation: {np.corrcoef(target_rf.flatten(), EI_diff.flatten())[0,1]:.4f}")


In [ ]:
# Fit dendritic neurons to all target RFs
results = {}

print("Fitting dendritic neurons to all target RFs...")
print("=" * 50)

for name, target_rf in targets.items():
    print(f"\nFitting to {name}...")
    E, I, diff, losses = fit_dendritic_neuron(target_rf, steps=600, lr=0.08, verbose=False)
    
    # Compute metrics
    final_loss = losses[-1]
    correlation = np.corrcoef(target_rf.flatten(), diff.flatten())[0,1]
    
    results[name] = {
        'target': target_rf,
        'E_weights': E,
        'I_weights': I, 
        'fitted_rf': diff,
        'final_loss': final_loss,
        'correlation': correlation,
        'losses': losses
    }
    
    print(f"  Final loss: {final_loss:.6f}")
    print(f"  Correlation: {correlation:.4f}")

print("\n" + "=" * 50)
print("Summary of all fits:")
for name, result in results.items():
    print(f"  {name:20s}: Loss={result['final_loss']:.6f}, Corr={result['correlation']:.4f}")

# Visualize all results
fig, axes = plt.subplots(3, len(results), figsize=(4*len(results), 12))

for i, (name, result) in enumerate(results.items()):
    vmax = np.abs(result['target']).max()
    
    # Target RF
    axes[0, i].imshow(result['target'], cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[0, i].set_title(f'Target\n{name}', fontsize=10)
    axes[0, i].axis('off')
    
    # Fitted RF (E-I difference)
    axes[1, i].imshow(result['fitted_rf'], cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[1, i].set_title(f'Fitted RF\nCorr={result["correlation"]:.3f}', fontsize=10)
    axes[1, i].axis('off')
    
    # Training curve
    axes[2, i].plot(result['losses'], 'b-', linewidth=1.5)
    axes[2, i].set_xlabel('Step')
    axes[2, i].set_ylabel('Loss')
    axes[2, i].set_title(f'Training\nFinal={result["final_loss"]:.2e}', fontsize=10)
    axes[2, i].grid(True, alpha=0.3)

plt.suptitle('Dendritic Neurons Fitted to Visual System RFs', fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
def test_orientation_tuning(fitted_rf, test_orientations):
    """Test orientation selectivity by correlating fitted RF with test gratings."""
    responses = []
    for ori in test_orientations:
        test_grating = rf_gabor(size=fitted_rf.shape[0], theta=np.deg2rad(ori))
        response = np.sum(fitted_rf * test_grating)
        responses.append(response)
    return np.array(responses)

def test_size_tuning(fitted_rf, spot_sizes):
    """Test size tuning with circular spots of different sizes."""
    responses = []
    center = fitted_rf.shape[0] // 2
    
    for size in spot_sizes:
        # Create circular spot
        y, x = np.ogrid[:fitted_rf.shape[0], :fitted_rf.shape[1]]
        mask = (x - center)**2 + (y - center)**2 <= (size/2)**2
        spot = np.zeros_like(fitted_rf)
        spot[mask] = 1.0
        
        response = np.sum(fitted_rf * spot)
        responses.append(response)
    return np.array(responses)

# Test orientation tuning for Gabor cells
print("Testing orientation tuning for fitted Gabor cells...")
test_angles = np.linspace(0, 180, 13, endpoint=False)

gabor_results = {}
for name, result in results.items():
    if 'Gabor' in name:
        responses = test_orientation_tuning(result['fitted_rf'], test_angles)
        osi = orientation_selectivity_index(responses, test_angles)
        gabor_results[name] = {'responses': responses, 'osi': osi}
        print(f"  {name}: OSI = {osi:.3f}")

# Plot orientation tuning curves
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Tuning curves
for name, data in gabor_results.items():
    axes[0].plot(test_angles, data['responses'], 'o-', label=f"{name} (OSI={data['osi']:.2f})", linewidth=2)

axes[0].set_xlabel('Orientation (degrees)')
axes[0].set_ylabel('Response')
axes[0].set_title('Orientation Tuning Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Polar plot
ax_polar = plt.subplot(122, projection='polar')
for name, data in gabor_results.items():
    theta = np.deg2rad(np.concatenate([test_angles, [180]]))
    responses_norm = data['responses'] / data['responses'].max()
    responses_complete = np.concatenate([responses_norm, [responses_norm[0]]])
    ax_polar.plot(theta, responses_complete, 'o-', label=name, linewidth=2)

ax_polar.set_title('Polar Tuning Plot')
ax_polar.legend(loc='upper left', bbox_to_anchor=(1.1, 1))

plt.tight_layout()
plt.show()

# Test size tuning for center-surround cells
print("\nTesting size tuning for center-surround cells...")
spot_sizes = np.linspace(2, 20, 10)

cs_results = {}
for name, result in results.items():
    if 'center' in name:
        responses = test_size_tuning(result['fitted_rf'], spot_sizes)
        si = size_tuning_index(responses)
        cs_results[name] = {'responses': responses, 'si': si}
        print(f"  {name}: Size Index = {si:.3f}")

# Plot size tuning curves
plt.figure(figsize=(8, 5))
for name, data in cs_results.items():
    plt.plot(spot_sizes, data['responses'], 'o-', label=f"{name} (SI={data['si']:.2f})", linewidth=2)

plt.xlabel('Spot Diameter')
plt.ylabel('Response')
plt.title('Size Tuning Curves (Center-Surround Cells)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\nBiological plausibility summary:")
print("- Gabor cells show sharp orientation tuning (OSI > 0.5 is typical for V1)")
print("- Center-surround cells exhibit size tuning with surround suppression")
print("- All responses emerge from pure E/I dendritic computation with divisive normalization")
